# Chapter 24: Case Study — Coastal Dynamics

*Part V — Domain Modeling: Land Use & Coastal Systems*

Implemented by the [`brmangue-dissmodel`](https://github.com/DisSModel/brmangue-dissmodel) package.

<div class="admonition warning">
<p class="admonition-title">Watch out</p>
<p>Like Chapter 23, this chapter is a draft written ahead of the underlying package settling. Specific performance figures below are illustrative of the kind of gap Chapter 21 already measured on <code>FireModel</code>, not numbers re-verified against <code>brmangue-dissmodel</code> directly in this sandbox — treat them as a placeholder to replace with a real benchmark once this chapter is revised.</p>
</div>

## Learning Objectives

By the end of this chapter you will be able to:

- Describe the coupled flood-propagation and mangrove-migration model this case study is built on
- Compare its raster and vector implementations, substrate for substrate
- Explain why flood propagation needs a push-based algorithm rather than `CellularAutomaton`'s pull-based `rule(idx)`
- Know the two independent ways this model validates itself

In [ ]:
# Standard imports — add chapter-specific imports below
import numpy as np
import matplotlib.pyplot as plt

Every substrate comparison in this book so far — Chapter 18's Game of Life, Chapter 21's `FireModel` — used a toy or synthetic case to make the vector-versus-raster point. This chapter closes Part V with the ecosystem's deepest real one: `brmangue-dissmodel`, a coupled model of coastal flooding and mangrove migration, implemented on both substrates deliberately, then checked against each other and against the original TerraME reference at once.

## The BR-MANGUE Model

`brmangue-dissmodel` implements the spatially explicit coastal ecosystem model from Bezerra et al. (2013), coupling two processes on the same grid:

1. **Flood dynamics** — sea-level rise propagating inland, with terrain elevation adjusting the water's reach cell by cell.
2. **Mangrove migration** — the ecosystem's response to that rising water: soil-state transitions and sediment accretion tracking where the mangrove can and can't persist.

Both processes share one `RasterBackend` (or one `GeoDataFrame`, on the vector substrate), coordinated within a single `Environment` — the same `SyncRasterModel` pattern Chapter 18 introduced for keeping two models' reads and writes consistent within one tick.

## Flood Propagation: A Push Model, Not a Pull Rule

Every cellular automaton this book has built since Chapter 21 follows `CellularAutomaton.rule(idx)`'s pull contract: each cell asks its neighbors for their state and decides its own next value. Flood propagation doesn't fit that shape naturally — water doesn't wait for a cell to ask "am I now flooded"; it originates at a source (the rising sea) and pushes outward, cell by cell, as elevation permits.

`brmangue-dissmodel`'s flood model is built directly on `SpatialModel` (or `RasterModel`) with a free-form `execute()`, not on `CellularAutomaton` — exactly the case Chapter 18 flagged when introducing the substrate classes: `CellularAutomaton`'s stricter `rule(idx)` contract is the right choice when a transition genuinely depends only on a cell's own neighbors, and the wrong one whenever a process spreads *from* a source outward instead. A push-based flood algorithm, faithful to the original TerraME implementation, is this book's clearest example of that second case.

## Two Substrates, Deliberately

`brmangue-dissmodel` ships genuinely parallel implementations, not one canonical version with an afterthought port:

- **Raster** (`brmangue.models.raster`) — NumPy/`RasterBackend`, vectorized, the fast path. This is the version validated directly against TerraME's own golden-output CSVs.
- **Vector** (`brmangue.models.vector`) — `GeoDataFrame`/`libpysal`, cell-by-cell over real polygon geometry. Numerically equivalent to the raster version, confirmed by a dedicated benchmark executor rather than assumed.

Both implement intentionally identical equations, thresholds, parameter names, and update ordering — the substrate is the only thing that differs, exactly the "same conceptual model, two substrates" pattern Chapter 18 first previewed and Chapter 21 measured concretely on `FireModel`.

## Performance: The Same Gap, at Domain Scale

Chapter 21 measured roughly a thousand-fold speedup moving `FireModel` from vector to raster on a small synthetic grid. BR-MANGUE's own raster-versus-vector gap follows the identical shape for the identical reason — per-cell Python method calls (vector) against a single vectorized NumPy pass (raster) — just at the scale a real coastal study area actually needs: a domain large enough to matter scientifically pushes the vector substrate's *linear-in-cell-count* cost from "slower" to "impractical," while the raster substrate's cost stays governed by array operations that scale far more gently. This is precisely why the raster implementation, not the vector one, is the version validated directly against TerraME's golden output — it's the version meant for a real production run.

## Validation: Two Independent Checks

BR-MANGUE validates itself two distinct ways, and both need to pass — neither alone is sufficient:

- **Raster vs. vector**, a benchmark executor running both substrates on the same input and reporting match percentage, mean absolute error, and root-mean-square error per output band — the same benchmark pattern Chapter 23 described for `disslucc-continuous`.
- **Raster vs. TerraME**, a stricter check comparing the raster substrate's output directly against TerraME's own golden-output CSVs at fixed checkpoints through a run — not "close to our own vector version," but "close to the original reference implementation itself."

Both checks are necessarily tolerance-based, not exact-match. Chapter 23's `disslucc-discrete` could demand 100% exact parity specifically because land-use class is categorical — a cell either matches its reference label or it doesn't. BR-MANGUE's state (elevation, tidal height, sediment) is continuous, so "correct" here means "within tolerance," the same way Chapter 23's `disslucc-continuous` validation worked, not the all-or-nothing bar `disslucc-discrete` could set for itself.

## Exercises

1. **Push versus pull, concretely.** Using *Flood Propagation*'s distinction, sketch — in words, no code needed — why a pull-based `rule(idx)` implementation of sea-level rise would need every cell to somehow "know" whether water has reached any upstream neighbor yet, and why that's awkward to express as a purely local rule.
2. **Which substrate for which purpose.** Given *Two Substrates, Deliberately* and *Performance*, when would you reach for the vector implementation despite its performance cost, and when would raster be the only practical choice? Answer using Chapter 21's own substrate-choice guidance, not intuition alone.
3. **Why two validation checks, not one.** Explain, in your own words, what a model could get wrong that a raster-vs-vector benchmark alone would catch but a raster-vs-TerraME comparison alone would miss, or vice versa.
4. **Tolerance, again.** Chapter 23 distinguished tolerance-based validation (continuous output) from exact-parity validation (categorical output). Which category does BR-MANGUE's flood/mangrove state fall into, and why does that make a 100%-parity bar the wrong target here even in principle?

In [ ]:
# Your code here

## Summary

### Key concepts introduced

- BR-MANGUE (Bezerra et al., 2013) as the ecosystem's deepest validation case: one coupled flood/mangrove-migration model, two deliberately parallel substrates, two independent checks
- Push-based flood propagation as the clearest example in this book of a process that doesn't fit `CellularAutomaton`'s pull-based `rule(idx)` contract, and why `SpatialModel`'s free-form `execute()` is the right base class for it instead
- The same vector-vs-raster performance gap Chapter 21 measured on `FireModel`, now at the scale a real coastal study domain actually requires
- Two validation checks, run independently — raster vs. vector (tolerance-based benchmark) and raster vs. TerraME (golden-output comparison) — neither substituting for the other
- Continuous state (elevation, tide, sediment) requiring tolerance-based validation, in deliberate contrast to Chapter 23's categorical, exact-parity `disslucc-discrete` case

This closes Part V. Part VI picks the story back up from a different angle — not a specific domain model, but the infrastructure (reproducibility, the platform, data cubes) every model in Parts IV and V ultimately runs on top of.

## Further Reading

- Bezerra, M. O. et al. (2013). Coastal ecosystem process modeling underlying BR-MANGUE — the primary reference this case study implements
- brmangue-dissmodel on GitHub: <https://github.com/DisSModel/brmangue-dissmodel>
- brmangue-qgis, the QGIS client for this model already introduced in Chapter 26: <https://github.com/DisSModel/brmangue-qgis>